---
title: "Crear circuitos"
description: "Cómo construir y visualizar circuitos cuánticos en Qiskit."
---

<span id="construct-circuits" />

# Crear circuitos



{/*
  DO NOT EDIT THIS CELL!!!
  This cell's content is generated automatically by a script. Anything you add
  here will be removed next time the notebook is run. To add new content, create
  a new cell before or after this one.
*/}

<details>
  <summary><b>Versiones del paquete</b></summary>

  El código de esta página se ha desarrollado teniendo en cuenta los siguientes requisitos.
  Recomendamos utilizar estas versiones o versiones más recientes.

  ```
  qiskit[all]~=2.3.0
  ```
</details>



Esta página echa un vistazo más de cerca a la clase [`QuantumCircuit`](/docs/api/qiskit/qiskit.circuit.QuantumCircuit) en el SDK de Qiskit, incluyendo algunos métodos más avanzados que puedes utilizar para crear circuitos cuánticos.



<span id="what-is-a-quantum-circuit" />

## ¿Qué es un circuito cuántico?

Un circuito cuántico simple es una colección de qubits y una lista de instrucciones que actúan sobre esos qubits. Para demostrarlo, la siguiente celda crea un nuevo circuito con dos nuevos qubits y, a continuación, muestra el atributo [`qubits`](/docs/api/qiskit/qiskit.circuit.QuantumCircuit#qubits) del circuito, que es una lista de [`Qubits`](/docs/api/qiskit/circuit#qiskit.circuit.Qubit) en orden desde el bit menos significativo $q_0$ hasta el bit más significativo $q_n$.



In [1]:
from qiskit import QuantumCircuit

qc = QuantumCircuit(2)
qc.qubits

[<Qubit register=(2, "q"), index=0>, <Qubit register=(2, "q"), index=1>]

Se pueden combinar varios objetos `QuantumRegister` y `ClassicalRegister` para crear un circuito. Cada [`QuantumRegister`](/docs/api/qiskit/circuit#qiskit.circuit.QuantumRegister) y [`ClassicalRegister`](/docs/api/qiskit/circuit#qiskit.circuit.ClassicalRegister) también pueden nombrarse.



In [2]:
from qiskit.circuit import QuantumRegister, ClassicalRegister

qr1 = QuantumRegister(2, "qreg1")  # Create a QuantumRegister with 2 qubits
qr2 = QuantumRegister(1, "qreg2")  # Create a QuantumRegister with 1 qubit
cr1 = ClassicalRegister(3, "creg1")  # Create a ClassicalRegister with 3 cbits

combined_circ = QuantumCircuit(
    qr1, qr2, cr1
)  # Create a quantum circuit with 2 QuantumRegisters and 1 ClassicalRegister
combined_circ.qubits

[<Qubit register=(2, "qreg1"), index=0>,
 <Qubit register=(2, "qreg1"), index=1>,
 <Qubit register=(1, "qreg2"), index=0>]

Puedes encontrar el índice y el registro de un qubit utilizando el método del circuito [`find_bit`](/docs/api/qiskit/qiskit.circuit.QuantumCircuit#qiskit.circuit.QuantumCircuit.find_bit) y sus atributos.



In [3]:
desired_qubit = qr2[0]  # Qubit 0 of register 'qreg2'

print("Index:", combined_circ.find_bit(desired_qubit).index)
print("Register:", combined_circ.find_bit(desired_qubit).registers)

Index: 2
Register: [(QuantumRegister(1, 'qreg2'), 0)]


Al añadir una instrucción al circuito, ésta se añade al atributo [`data`](/docs/api/qiskit/qiskit.circuit.QuantumCircuit#data) del circuito. La siguiente salida de celda muestra que `data` es una lista de [`CircuitInstruction`](/docs/api/qiskit/qiskit.circuit.CircuitInstruction) objetos, cada uno de los cuales tiene un atributo `operation` , y un atributo `qubits` .



In [4]:
qc.x(0)  # Add X-gate to qubit 0
qc.data

[CircuitInstruction(operation=Instruction(name='x', num_qubits=1, num_clbits=0, params=[]), qubits=(<Qubit register=(2, "q"), index=0>,), clbits=())]

La forma más sencilla de ver esta información es a través del método [`draw`](/docs/api/qiskit/qiskit.circuit.QuantumCircuit#draw) que devuelve una visualización de un circuito. Consulte [Visualizar circuitos](./visualize-circuits) para conocer distintas formas de mostrar circuitos cuánticos.



In [5]:
qc.draw("mpl")

<Image src="/docs/images/guides/construct-circuits/extracted-outputs/43a57258-3e33-4071-8a48-2bf127c8a5be-0.svg" alt="Output of the previous code cell" />

Los objetos de instrucción de circuito pueden contener circuitos de "definición" que describen la instrucción en términos de instrucciones más fundamentales. Por ejemplo, la [puerta X](/docs/api/qiskit/qiskit.circuit.library.XGate) se define como un caso específico de la puerta [U3-gate](/docs/api/qiskit/qiskit.circuit.library.U3Gate) una puerta más general de un solo qubit.



In [6]:
# Draw definition circuit of 0th instruction in `qc`
qc.data[0].operation.definition.draw("mpl")

<Image src="/docs/images/guides/construct-circuits/extracted-outputs/653e2427-e301-4d2f-84de-1959185ace8e-0.svg" alt="Output of the previous code cell" />

Las instrucciones y los circuitos se parecen en que ambos describen operaciones con bits y qubits, pero tienen propósitos diferentes:

*   Las instrucciones se tratan como fijas, y sus métodos suelen devolver nuevas instrucciones (sin mutar el objeto original).
*   Los circuitos están diseñados para ser construidos sobre muchas líneas de código, y [`QuantumCircuit`](/docs/api/qiskit/qiskit.circuit.QuantumCircuit) los métodos a menudo mutan el objeto existente.



<span id="what-is-circuit-depth" />

### ¿Qué es la profundidad del circuito?

La [profundidad()](/docs/api/qiskit/qiskit.circuit.QuantumCircuit#qiskit.circuit.QuantumCircuit.depth) de un circuito cuántico es una medida del número de "capas" de puertas cuánticas, ejecutadas en paralelo, que se necesitan para completar el cálculo definido por el circuito. Dado que las puertas cuánticas tardan tiempo en implementarse, la profundidad de un circuito corresponde aproximadamente a la cantidad de tiempo que tarda el ordenador cuántico en ejecutar el circuito. Así, la profundidad de un circuito es una magnitud importante para medir si un circuito cuántico puede funcionar en un dispositivo.

El resto de esta página ilustra cómo manipular circuitos cuánticos.



<span id="build-circuits" />

## Construir circuitos

Métodos como [`QuantumCircuit.h`](/docs/api/qiskit/qiskit.circuit.QuantumCircuit#h) y [`QuantumCircuit.cx`](/docs/api/qiskit/qiskit.circuit.QuantumCircuit#cx) añaden instrucciones específicas a los circuitos. Para añadir instrucciones a un circuito de forma más general, utilice el método [`append`](/docs/api/qiskit/qiskit.circuit.QuantumCircuit#append) método. Toma una instrucción y una lista de qubits a los que aplicar la instrucción. Consulte [la documentación de la API Circuit Library](/docs/api/qiskit/circuit_library) para obtener una lista de las instrucciones compatibles.



In [7]:
from qiskit.circuit.library import HGate

qc = QuantumCircuit(1)
qc.append(
    HGate(),  # New HGate instruction
    [0],  # Apply to qubit 0
)
qc.draw("mpl")

<Image src="/docs/images/guides/construct-circuits/extracted-outputs/66813cae-9841-47ea-96b7-8fd7b82e9759-0.svg" alt="Output of the previous code cell" />

Para combinar dos circuitos, utilice el método [`compose`](/docs/api/qiskit/qiskit.circuit.QuantumCircuit#compose) método. Este acepta otro [`QuantumCircuit`](/docs/api/qiskit/qiskit.circuit.QuantumCircuit) y una lista opcional de asignaciones de qubits.

<Admonition type="note">
  El método [`compose`](/docs/api/qiskit/qiskit.circuit.QuantumCircuit#compose) devuelve un circuito nuevo y **no** muta ninguno de los circuitos sobre los que actúa. Para mutar el circuito sobre el que se llama al método [`compose`](/docs/api/qiskit/qiskit.circuit.QuantumCircuit#compose) utilice el argumento `inplace=True`.
</Admonition>



In [8]:
qc_a = QuantumCircuit(4)
qc_a.x(0)

qc_b = QuantumCircuit(2, name="qc_b")
qc_b.y(0)
qc_b.z(1)

# compose qubits (0, 1) of qc_a to qubits (1, 3) of qc_b respectively
combined = qc_a.compose(qc_b, qubits=[1, 3])
combined.draw("mpl")

<Image src="/docs/images/guides/construct-circuits/extracted-outputs/29152dfa-2275-4bc4-aadb-82185b9e0e86-0.svg" alt="Output of the previous code cell" />

También puedes compilar los circuitos en instrucciones para mantenerlos organizados. Puede convertir un circuito en una instrucción utilizando el método [`to_instruction`](/docs/api/qiskit/qiskit.circuit.QuantumCircuit#to_instruction) y añadirla a otro circuito como lo haría con cualquier otra instrucción. El circuito dibujado en la celda siguiente es funcionalmente equivalente al circuito dibujado en la celda anterior.



In [9]:
inst = qc_b.to_instruction()
qc_a.append(inst, [1, 3])
qc_a.draw("mpl")

<Image src="/docs/images/guides/construct-circuits/extracted-outputs/81b682dd-45cb-4492-809e-d9e8ebbf5600-0.svg" alt="Output of the previous code cell" />

Si tu circuito es unitario, puedes convertirlo en un [`Gate`](/docs/api/qiskit/qiskit.circuit.Gate) utilizando el método [`to_gate`](/docs/api/qiskit/qiskit.circuit.QuantumCircuit#to_gate) método. [`Gate`](/docs/api/qiskit/qiskit.circuit.Gate) son tipos específicos de instrucciones que tienen algunas características adicionales, como el [`control`](/docs/api/qiskit/qiskit.circuit.Gate#control) que añade un control cuántico.



In [10]:
gate = qc_b.to_gate().control()
qc_a.append(gate, [0, 1, 3])
qc_a.draw("mpl")

<Image src="/docs/images/guides/construct-circuits/extracted-outputs/ed362e64-d6a4-4dfd-a5cf-5e6bdc7a81b5-0.svg" alt="Output of the previous code cell" />

Para ver lo que está pasando, puede utilizar el método [`decompose`](/docs/api/qiskit/qiskit.circuit.QuantumCircuit#decompose) para expandir cada instrucción en su definición.

<Admonition type="note">
  El método [`decompose`](/docs/api/qiskit/qiskit.circuit.QuantumCircuit#decompose) devuelve un nuevo circuito y **no** muta el circuito sobre el que actúa.
</Admonition>



In [11]:
qc_a.decompose().draw("mpl")

<Image src="/docs/images/guides/construct-circuits/extracted-outputs/3c0633db-929b-4428-a888-7a3d493bd6dd-0.svg" alt="Output of the previous code cell" />

<span id="measure-qubits" />

<span id="measure-qubits" />

## Medir qubits

Las mediciones se utilizan para muestrear los estados de los qubits individuales y transferir los resultados a un registro clásico. Tenga en cuenta que si presenta circuitos a una primitiva [Sampler](./primitives#sampler), se requieren mediciones. Sin embargo, los circuitos enviados a una primitiva de [Estimador](./primitives#estimator) no deben contener mediciones.

Los qubits pueden medirse por tres métodos: [`measure`](/docs/api/qiskit/qiskit.circuit.QuantumCircuit#qiskit.circuit.QuantumCircuit.measure), [`measure_all`](/docs/api/qiskit/qiskit.circuit.QuantumCircuit#measure_all) y [`measure_active`](/docs/api/qiskit/qiskit.circuit.QuantumCircuit#measure_active). Para saber cómo visualizar los resultados medidos, consulte la página [Visualizar resultados](./visualize-results).

1.  `QuantumCircuit.measure` mide cada qubit en el primer argumento sobre el bit clásico dado como segundo argumento. Este método permite controlar totalmente dónde se almacena el resultado de la medición.

2.  `QuantumCircuit.measure_all` no necesita argumento y puede utilizarse para circuitos cuánticos sin bits clásicos predefinidos. Crea hilos clásicos y almacena por orden los resultados de las mediciones. Por ejemplo, la medición del qubit $q_i$ se almacena en el cbit $meas_i$ ). También añade una barrera antes de la medición.

3.  `QuantumCircuit.measure_active` : similar a `measure_all`, pero sólo mide los qubits que tienen operaciones.



In [12]:
qc1 = QuantumCircuit(2, 2)
qc1.measure(0, 1)
qc1.draw("mpl", cregbundle=False)

<Image src="/docs/images/guides/construct-circuits/extracted-outputs/0cdb2273-0.svg" alt="Output of the previous code cell" />

In [13]:
qc2 = QuantumCircuit(2)
qc2.measure_all()
qc2.draw("mpl", cregbundle=False)

<Image src="/docs/images/guides/construct-circuits/extracted-outputs/6f33698c-0.svg" alt="Output of the previous code cell" />

In [14]:
qc3 = QuantumCircuit(2)
qc3.x(1)
qc3.measure_active()
qc3.draw("mpl", cregbundle=False)

<Image src="/docs/images/guides/construct-circuits/extracted-outputs/ca3f225f-0.svg" alt="Output of the previous code cell" />

<span id="parameterized-circuits" />

## Circuitos parametrizados

Muchos algoritmos cuánticos a corto plazo implican la ejecución de muchas variaciones de un circuito cuántico. Dado que la construcción y optimización de circuitos de gran tamaño puede resultar costosa desde el punto de vista informático, Qiskit admite circuitos **parametrizados**. Estos circuitos tienen parámetros indefinidos, y no es necesario definir sus valores hasta justo antes de ejecutar el circuito. Esto le permite mover la construcción del circuito y la optimización fuera del bucle principal del programa.  La siguiente celda crea y muestra un circuito parametrizado.



In [15]:
from qiskit.transpiler import generate_preset_pass_manager
from qiskit.circuit import Parameter

angle = Parameter("angle")  # undefined number

# Create and optimize circuit once
qc = QuantumCircuit(1)
qc.rx(angle, 0)
qc = generate_preset_pass_manager(
    optimization_level=3, basis_gates=["u", "cx"]
).run(qc)

qc.draw("mpl")

<Image src="/docs/images/guides/construct-circuits/extracted-outputs/a580552c-d585-4047-99f0-32aafd06e4f3-0.svg" alt="Output of the previous code cell" />

La siguiente celda crea muchas variaciones de este circuito y muestra una de las variaciones.



In [16]:
circuits = []
for value in range(100):
    circuits.append(qc.assign_parameters({angle: value}))

circuits[0].draw("mpl")

<Image src="/docs/images/guides/construct-circuits/extracted-outputs/85af6231-921a-4130-99d3-f6998f761df8-0.svg" alt="Output of the previous code cell" />

Encontrará una lista de los parámetros no definidos de un circuito en su atributo `parameters` .



In [17]:
qc.parameters

ParameterView([Parameter(angle)])

<span id="change-a-parameters-name" />

### Cambiar el nombre de un parámetro

Por defecto, los nombres de los parámetros de un circuito parametrizado llevan el prefijo `x`-, por ejemplo, `x[0]`. Puede cambiar los nombres después de haberlos definido, como se muestra en el siguiente ejemplo.



In [18]:
from qiskit.circuit.library import z_feature_map
from qiskit.circuit import ParameterVector

# Define a parameterized circuit with default names
# For example, x[0]
circuit = z_feature_map(2)

# Set new parameter names
# They will now be prefixed by `hi` instead
# For example, hi[0]
training_params = ParameterVector("hi", 2)

# Assign parameter names to the quantum circuit
circuit = circuit.assign_parameters(parameters=training_params)

<CodeAssistantAdmonition
  tagLine="Forgotten the method name? Try asking Qiskit Code Assistant."
  prompts={[
  "# Assign all parameters in qc to 0"
  ]}
/>



<span id="next-steps" />

## Próximos pasos

<Admonition type="tip" title="Recomendaciones">
  *   Para aprender sobre algoritmos cuánticos a corto plazo, siga el curso [Diseño de algoritmos variacionales](/learning/courses/variational-algorithm-design).
  *   Vea un ejemplo de circuitos utilizados en el tutorial [del Algoritmo de Grover](/docs/tutorials/grovers-algorithm).
  *   Trabaje con circuitos sencillos utilizando [IBM Quantum Composer](/docs/guides/composer).
</Admonition>



© IBM Corp., 2017-2026